# Cartographie Automatisée des Unités Géologiques par IA

## Introduction
La cartographie géologique traditionnelle est un processus long et coûteux. Ce notebook démontre comment l'IA **Prithvi EO v2** peut accélérer ce travail en segmentant automatiquement le terrain en unités cohérentes basées sur la lithologie, la texture et le contexte géomorphologique.

## Objectifs
*   **Segmentation régionale** : Diviser la zone d'étude en domaines géologiques homogènes.
*   **Extraction de caractéristiques** : Utiliser un encodeur IA pour capturer les nuances rocheuses invisibles à l'œil nu.
*   **Visualisation structurale** : Créer une carte géologique de base pour l'exploration minérale.

## Méthodologie
1.  **Setup** : Installation de TerraTorch.
2.  **Acquisition** : Images Sentinel-2 multispectrales.
3.  **Inférence Prithvi** : Génération de descripteurs sémantiques haute dimension.
4.  **Clustering Hiérarchique** : Regroupement des signatures par la méthode de Ward pour former les unités.

In [ ]:
# ====================================================
# ÉTAPE 1 : Setup
# ====================================================
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib seaborn -q

import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from terratorch import BACKBONE_REGISTRY

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print("✅ Système prêt")

## Zone d'Étude (ROI)
Utilisation du rectangle régional défini pour l'étude Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d\'étude')
Map

## Acquisition des Données Satellite
Exportation des bandes optiques et infrarouges.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).median().clip(roi))

geemap.ee_export_image(image.select(['B2','B3','B4','B8','B11','B12']), 'geo_units.tif', scale=30, region=roi)

## Inférence et Segmentation en Unités
Nous utilisons le clustering hiérarchique avec la méthode de Ward pour forcer la création de 8 unités géologiques majeures basées sur les caractéristiques complexes de Prithvi.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence et Clustering Hiérarchique
# ====================================================
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('geo_units.tif') as src: img = src.read().astype(np.float32) / 10000.0

with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
side = int(np.sqrt(feats_np.shape[0]))

dist_matrix = pdist(feats_np, metric='cosine')
linkage_matrix = linkage(dist_matrix, method='ward')
units = fcluster(linkage_matrix, 8, criterion='maxclust')
unit_map = units.reshape(side, side)

plt.figure(figsize=(10, 8))
plt.imshow(unit_map, cmap='terrain')
plt.colorbar(label='Unités Géologiques identifiées')
plt.title("Carte des Unités Géologiques (Ward + Prithvi)")
plt.axis('off')
plt.show()